# 04 · ESM1b — a protein language model

ESM1b (Brandes et al. 2023, *Nat Genet*, PMID 37563329) is an *LLM for protein sequences*. Like EVE it is **unsupervised**, but it scores variants on a **log-likelihood ratio (LLR)** whose scale runs *backwards* from every other tool here.

> ### The 3 CFTR UniProt IDs (P13569, P13569-2, P13569-3)
> ESM1b variant files (ntranoslab) list **three** CFTR-related isoforms:
> **P13569** is the **canonical** CFTR isoform (1480 aa; matches MANE `NM_000492.4`),
> while **-2** and **-3** are alternative UniProt isoforms (differ by alternative
> splicing). **Use the canonical `P13569`** so residue numbering lines up with
> AlphaMissense / CFTR2 / gnomAD — picking `-2`/`-3` silently shifts positions and
> breaks the `protein_variant` join. *(Isoform details: UniProt P13569.)*

> ✅ **REAL DATA.** Full CFTR **saturation** ESM1b LLR — **~28,120** variants (all 1,480 residues), `data/esm1b_cftr.csv`, built by the cell below from a manually-downloaded release zip (ntranoslab, canonical UniProt **P13569**). `source == 'REAL'`.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 1 · ESM1b — a protein language model

**What is it?** ESM1b is a **protein language model** — think of it as an *LLM for protein sequences*. Just as a text LLM learns which word is likely to come next, ESM1b was trained on millions of natural protein sequences to predict which amino acid "belongs" at each position given its neighbours. It never saw the alignment explicitly; it learned protein grammar directly from raw sequences.

To score a variant, ESM1b compares how *likely* the model thinks the **mutant** amino acid is versus the **wild-type** amino acid at that position. This is a **log-likelihood ratio (LLR)**:

$$\text{LLR} = \log \frac{P(\text{mutant amino acid})}{P(\text{wild-type amino acid})}$$

> ### 🔄 IMPORTANT — ESM1b runs *backwards* from the other tools
> A **more negative** LLR means the model finds the mutation more *surprising* → **more damaging**. This is the **opposite direction** to EVE, AlphaMissense, and REVEL, where *higher* = worse.
> 
> - **Cut-off:** `LLR <= -7.5` ~ pathogenic. **Lower (more negative) = worse.**
> - Unsupervised (learned from sequences only) → low circularity, just like EVE.

## Building the REAL data — a manual download (bulk release, no per-gene API)

ESM1b has no API — the ntranoslab team publishes per-isoform LLR matrices for
**every human protein** (~42,000 files) as one zip. **You must fetch this one
yourself:**

1. Go to the HuggingFace Space **`ntranoslab/esm_variants`**
   (github.com/ntranoslab/esm-variants) and download
   **`ALL_hum_isoforms_ESM1b_LLR.zip`**.
2. Save it as `data/ALL_hum_isoforms_ESM1b_LLR.zip` (gitignored — never commit it).

The cell below opens the zip **without extracting it**, reads only CFTR's one
member (`P13569_LLR.csv` — **P13569 is the canonical isoform**, see the callout
above), melts its wide LLR matrix (columns = wild-type+position, rows = mutant
amino acid) into the long `protein_variant` form every other tool uses, and
writes `data/esm1b_cftr.csv`. Unlike EVE, this is **full saturation**: all 1,480
residues × 19 substitutions (~28,120 rows) — the matrix has no missing cells.

License: code is MIT; the scores themselves are released "per publication"
(Brandes et al. 2023) rather than under an explicit redistribution license —
see `data_manifest.json`.

In [2]:
import zipfile, io

DATA_DIR = pathlib.Path.cwd().parent / "data"
ESM1B_ZIP = DATA_DIR / "ALL_hum_isoforms_ESM1b_LLR.zip"
ESM1B_MEMBER = "content/ALL_hum_isoforms_ESM1b_LLR/P13569_LLR.csv"
ESM1B_TSV = DATA_DIR / "esm1b_cftr.csv"

if ESM1B_TSV.exists():
    print(f"already built -> {ESM1B_TSV.name} (delete to rebuild)")
elif not ESM1B_ZIP.exists():
    raise FileNotFoundError(
        f"{ESM1B_ZIP} not found.\n"
        "ESM1b has no per-gene API -- get the bulk release (one <UniProt>_LLR.csv per\n"
        "human isoform, ~42,000 files, packaged as one zip):\n"
        "  1. Go to the HuggingFace Space 'ntranoslab/esm_variants'\n"
        "     (github.com/ntranoslab/esm-variants) and download\n"
        "     'ALL_hum_isoforms_ESM1b_LLR.zip'\n"
        f"  2. Save it as {ESM1B_ZIP} (do NOT commit it -- data/ is gitignored)\n"
        "Then re-run this cell -- it reads only CFTR's one member file (P13569_LLR.csv)\n"
        "from inside the zip, never extracting the other ~42,000."
    )
else:
    with zipfile.ZipFile(ESM1B_ZIP) as z:
        with z.open(ESM1B_MEMBER) as fh:
            m = pd.read_csv(io.TextIOWrapper(fh, encoding="utf-8"), index_col=0)
    # m is an LLR matrix: columns "<wt> <pos>" (e.g. 'M 1'), rows = mutant amino acid.
    rows = []
    for col in m.columns:
        wt, pos = col.split(" ")
        pos = int(pos)
        for mut in m.index:
            if mut == wt:
                continue
            val = m.at[mut, col]
            if pd.isna(val):
                continue
            rows.append((f"{wt}{pos}{mut}", wt, pos, mut, round(float(val), 4)))
    df = pd.DataFrame(rows, columns=["protein_variant", "wt_aa", "position", "mt_aa", "esm1b_score"])
    df = df.sort_values("position").reset_index(drop=True)
    df["source"] = "REAL"
    df.to_csv(ESM1B_TSV, index=False)
    print(f"REAL ESM1b CFTR variants written: {len(df):,} -> {ESM1B_TSV.relative_to(DATA_DIR.parent)}")

already built -> esm1b_cftr.csv (delete to rebuild)


In [3]:
esm = tk.load_esm1b()      # REAL — full CFTR saturation LLR (~28,120), built by the cell above
print(f"{len(esm):,} REAL ESM1b variants | source: {esm['source'].unique().tolist()}")
print('LLR range:', esm['esm1b_score'].min(), '->', esm['esm1b_score'].max(),
      '| pathogenic (<= -7.5):', int((esm['esm1b_score'] <= -7.5).sum()))
esm.head(8)

28,120 REAL ESM1b variants | source: ['REAL']
LLR range: -23.793 -> 4.943 | pathogenic (<= -7.5): 14948


,protein_variant,esm1b_score,source
0,M1K,-5.762,REAL
1,M1R,-6.544,REAL
2,M1H,-8.114,REAL
3,M1E,-6.197,REAL
4,M1D,-6.949,REAL
5,M1N,-6.897,REAL
6,M1Q,-7.112,REAL
7,M1T,-6.903,REAL


## 2 · Turn an ESM1b LLR into a call

`tk.call_from_score(score, 'esm1b')` bakes in the **direction**: cut at `<= -7.5`, and **lower / more negative = worse**. You do not juggle the sign yourself.

In [4]:
esm['esm1b_call'] = esm['esm1b_score'].apply(lambda s: tk.call_from_score(s, 'esm1b'))
esm[['protein_variant', 'esm1b_score', 'esm1b_call', 'source']].head(12)

,protein_variant,esm1b_score,esm1b_call,source
0,M1K,-5.762,benign,REAL
1,M1R,-6.544,benign,REAL
2,M1H,-8.114,pathogenic,REAL
3,M1E,-6.197,benign,REAL
4,M1D,-6.949,benign,REAL
5,M1N,-6.897,benign,REAL
6,M1Q,-7.112,benign,REAL
7,M1T,-6.903,benign,REAL
8,M1S,-6.292,benign,REAL
9,M1C,-8.059,pathogenic,REAL


### The sign flip, made concrete

It is worth staring at *one* variant until the backwards scale clicks. Run the next cell.

In [5]:
# Most negative ESM1b = most damaging.
row = esm.sort_values('esm1b_score').iloc[0]
print(f"Variant {row['protein_variant']}:")
print(f"  ESM1b = {row['esm1b_score']:>6}  ->  {row['esm1b_call']:<10} "
      f"(LOW / negative score, cut at -7.5, so low = pathogenic)")
print('A more negative LLR = a more surprising mutation = more damaging —')
print('the opposite direction to EVE, AlphaMissense and REVEL.')

Variant R289P:
  ESM1b = -23.793  ->  pathogenic (LOW / negative score, cut at -7.5, so low = pathogenic)
A more negative LLR = a more surprising mutation = more damaging —
the opposite direction to EVE, AlphaMissense and REVEL.


## Key takeaways

1. **ESM1b** scores the mutant-vs-wild-type **log-likelihood ratio**; cut `<= -7.5` ~ pathogenic — **lower / more negative = worse** (opposite to EVE). `tk.call_from_score` handles the sign.
2. **Unsupervised** → low circularity vs ClinVar (like EVE).
3. This notebook now uses **REAL ESM1b** — full CFTR saturation (~28,120 variants, P13569), `source == REAL`.

**Next:** tools/05 — **REVEL** (supervised → circularity).